In [ ]:
from __future__ import annotations

import ast
import json
import string
from pathlib import Path
from typing import Dict, List

# Paths inside reasoning/
COT_FILE = Path("cot_only_oat.py")
DATASET_PATH = Path("data/race_train_high_42_3.jsonl")


def load_exact_prompt_functions(cot_file: Path):
    """Load the exact prompt-related function definitions from cot_only_oat.py."""
    source = cot_file.read_text(encoding="utf-8")
    module = ast.parse(source)

    wanted = {
        "_mcq_block",
        "qa_cot_prompt",
        "parse_questions",
        "normalize_gold_letter",
    }

    fn_src = []
    for node in module.body:
        if isinstance(node, ast.FunctionDef) and node.name in wanted:
            seg = ast.get_source_segment(source, node)
            if seg:
                fn_src.append(seg)

    ns = {
        "Dict": Dict,
        "List": List,
        "json": json,
        "string": string,
    }
    exec("\n\n".join(fn_src), ns)

    return (
        ns["_mcq_block"],
        ns["qa_cot_prompt"],
        ns["parse_questions"],
        ns["normalize_gold_letter"],
    )


_mcq_block, qa_cot_prompt, parse_questions, normalize_gold_letter = load_exact_prompt_functions(COT_FILE)


def load_first_dataset_example(dataset_path: Path):
    """Load one exact raw example from your JSONL training dataset."""
    with dataset_path.open("r", encoding="utf-8") as f:
        first_line = f.readline().strip()
    if not first_line:
        raise ValueError(f"Dataset file is empty: {dataset_path}")
    return json.loads(first_line)


def build_prompt_from_dataset_example(example: dict, question_index: int = 0, qa_num_samples: int = 3):
    """
    Build prompts from a real dataset row using the exact cot_only_oat prompt code.
    This mirrors prompt formatting exactly and uses one selected valid question from the row.
    """
    article = example.get("article", "")
    questions = parse_questions(example.get("questions", []))

    valid = []
    for q in questions:
        q_text = q.get("question", "")
        opts = q.get("options", [])
        gold = normalize_gold_letter(q.get("answer", ""))
        if isinstance(opts, list) and len(opts) >= 4 and q_text and gold:
            valid.append({"question": q_text, "options": opts, "answer": gold})

    if not valid:
        return ["Question: N/A\nAnswer with \\boxed{A}."] * qa_num_samples, None

    idx = max(0, min(question_index, len(valid) - 1))
    picked = valid[idx]
    prompt_text = qa_cot_prompt(
        article=article,
        question_text=picked["question"],
        options=picked["options"],
    )
    return [prompt_text] * qa_num_samples, picked


# ---- Real example from dataset ----
example = load_first_dataset_example(DATASET_PATH)
examples, picked = build_prompt_from_dataset_example(example, question_index=0, qa_num_samples=3)

print(f"Dataset file: {DATASET_PATH}")
print(f"example_id: {example.get('example_id', 'N/A')}")
print(f"Generated {len(examples)} prompt(s).")
if picked is not None:
    print(f"Gold answer for selected question: {picked['answer']}")

print("\n=== Prompt example (exact text fed to model) ===\n")
print(examples[0])

print("\n=== Verify repetition across samples (as in actor.step) ===")
for i, p in enumerate(examples, start=1):
    print(f"Sample {i}: {p == examples[0]}")